# LangGraph：控制流、工具与故障
真实 LangGraph 1.2.14，CPU，无模型密钥。模型选择使用固定教学决策，不能把通过断言理解为模型效果。按节运行；先预测输出，再改变一个条件。检查点与跨进程恢复见另一个 durable-workflow notebook。

In [ ]:
import os, sys, subprocess
os.environ['LANGSMITH_TRACING']='false'
os.environ['LANGCHAIN_TRACING_V2']='false'
os.environ['LANGGRAPH_STRICT_MSGPACK']='true'
subprocess.check_call([sys.executable,'-m','pip','install','-q','langgraph==1.2.14','langgraph-checkpoint==4.2.0','langgraph-checkpoint-sqlite==3.1.1'])

## 1. 条件边与 Command
条件边按更新后的状态选一个分支。Command 同时更新与跳转，但已有静态边仍执行。反例会同时走两条边。

In [ ]:
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command, Send, RetryPolicy
from langgraph.errors import GraphRecursionError
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages

class RouteState(TypedDict):
    needs_search: bool
    visited: Annotated[list[str], add]
def choose(s):
    return Command(update={'visited':['choose']}, goto='search' if s['needs_search'] else 'answer')
def make_router(extra_static=False):
    b=StateGraph(RouteState)
    b.add_node('choose',choose)
    b.add_node('search',lambda s:{'visited':['search']})
    b.add_node('answer',lambda s:{'visited':['answer']})
    b.add_edge(START,'choose')
    if extra_static: b.add_edge('choose','answer')
    b.add_edge('search',END);b.add_edge('answer',END)
    return b.compile()
bad=make_router(True).invoke({'needs_search':True,'visited':[]})['visited']
good=make_router().invoke({'needs_search':True,'visited':[]})['visited']
assert set(bad)=={'choose','search','answer'} and good==['choose','search']
assert make_router().invoke({'needs_search':False,'visited':[]})['visited']==['choose','answer']
print('Command + 静态边:',bad,'; 独占分支:',good)
# 同 ID 消息更新，替换原消息，而不是追加一份。
merged=add_messages([HumanMessage('旧问题',id='q1')],[HumanMessage('新问题',id='q1')])
assert len(merged)==1 and merged[0].content=='新问题'
print('add_messages 同 ID:',merged[0].content)

## 2. Send 动态 fan-out 与归并
三个任务分别携带自己的输入；结果 reducer 只在内容一致时接受同 ID。排序用于展示，不能假设并发完成顺序。

In [ ]:
def merge_items(old,new):
    for k in old.keys() & new.keys():
        if old[k]!=new[k]: raise ValueError('same ID, different result')
    return {**old,**new}
class Batch(TypedDict):
    items: list[int]
    results: Annotated[dict[str,int],merge_items]
    total: int
    joins: int
def dispatch(s):
    return [Send('square',{'item':x}) for x in s['items']]
def square(s): return {'results':{str(s['item']):s['item']**2}}
def join(s): return {'total':sum(s['results'].values()),'joins':s['joins']+1}
b=StateGraph(Batch)
b.add_node('dispatch',lambda s:{})
b.add_node('square',square);b.add_node('join',join)
b.add_edge(START,'dispatch');b.add_conditional_edges('dispatch',dispatch,['square'])
b.add_edge('square','join');b.add_edge('join',END)
batch=b.compile().invoke({'items':[2,3,4],'results':{},'joins':0,'total':0})
assert batch['total']==29 and batch['joins']==1 and len(batch['results'])==3
print('fan-out:',batch)
# 本节是同一 super-step 的统一 fan-out；不同深度分支的 join 要另建屏障。

## 3. 真正的 ToolNode 循环
固定决策发出一个 typed tool call，ToolNode 实际执行计算器，ToolMessage 回到决策节点。更换为模型时仍保留这些协议与权限边界。

In [ ]:
@tool
def multiply(a:int,b:int)->int:
    """Multiply two integers without external side effects."""
    return a*b
def policy(s):
    if isinstance(s['messages'][-1],ToolMessage):
        return {'messages':[AIMessage(content='结果：'+s['messages'][-1].content)]}
    return {'messages':[AIMessage(content='',tool_calls=[{'name':'multiply','args':{'a':6,'b':7},'id':'call-1','type':'tool_call'}])]}
b=StateGraph(MessagesState)
b.add_node('policy',policy);b.add_node('tools',ToolNode([multiply]))
b.add_edge(START,'policy');b.add_conditional_edges('policy',tools_condition)
b.add_edge('tools','policy')
tool_output=b.compile().invoke({'messages':[HumanMessage('6 × 7')]},{'recursion_limit':8})
messages=tool_output['messages']
assert [m.type for m in messages]==['human','ai','tool','ai']
assert messages[2].tool_call_id=='call-1' and messages[2].content=='42'
assert messages[-1].content=='结果：42'
print('消息序列:',[(m.type,m.content) for m in messages])
# 工具名、参数 Schema、关联 ID 和结果语义分别检查；模型输出不是授权凭证。

## 4. 子图接口映射、流式更新与私有字段
父图和子图使用不同字段名。input/output schema 过滤 invoke 返回值，但不自动隐藏 stream 的内部字段；输出到 UI 前要显式白名单。

In [ ]:
class Child(TypedDict):
    raw: str
    normalized: str
child=StateGraph(Child).add_node('normalize',lambda s:{'normalized':s['raw'].strip().lower()}).add_edge(START,'normalize').add_edge('normalize',END).compile()
class Input(TypedDict): query:str
class Output(TypedDict): answer:str
class Parent(TypedDict):
    query:str
    internal_note:str
    answer:str
def wrap(s):
    result=child.invoke({'raw':s['query']})
    return {'answer':result['normalized'],'internal_note':'教学内部字段，无秘密'}
parent=StateGraph(Parent,input_schema=Input,output_schema=Output).add_node('wrap',wrap).add_edge(START,'wrap').add_edge('wrap',END).compile()
assert parent.invoke({'query':'  AGENT  '})=={'answer':'agent'}
events=list(parent.stream({'query':'  AGENT  '},stream_mode=['updates','values'],version='v2'))
assert any('internal_note' in e['data'] for e in events if e['type']=='values')
safe=list(parent.stream({'query':'  AGENT  '},stream_mode='values',output_keys=['answer'],version='v2'))
assert all(set(e['data'])<={'answer'} for e in safe)
print('未筛选最后快照:',[e['data'] for e in events if e['type']=='values'][-1])
print('白名单快照:',safe[-1]['data'])

## 5. 重试、循环预算与 pending writes
显式选择 ConnectionError 重试，不把参数或权限错误无限重试。步骤上限不是时间或费用上限。并行任务中成功写入的节点，在失败恢复时不再次执行。

In [ ]:
class Count(TypedDict): n:int
attempts={'n':0}
def unstable(s):
    attempts['n']+=1
    if attempts['n']<3: raise ConnectionError('temporary fixture')
    return {'n':s['n']+1}
retry=StateGraph(Count).add_node('unstable',unstable,retry_policy=RetryPolicy(initial_interval=0,jitter=False,max_attempts=3,retry_on=ConnectionError)).add_edge(START,'unstable').add_edge('unstable',END).compile()
assert retry.invoke({'n':0})['n']==1 and attempts['n']==3
loop=StateGraph(Count).add_node('loop',lambda s:{'n':s['n']+1}).add_edge(START,'loop').add_edge('loop','loop').compile()
try: loop.invoke({'n':0},{'recursion_limit':3})
except GraphRecursionError: print('无终止循环被步骤上限阻止')
else: raise AssertionError('loop did not stop')

class Parallel(TypedDict): results:Annotated[dict[str,int],merge_items]
calls={'good':0,'bad':0}
def succeeds(s):
    calls['good']+=1
    return {'results':{'good':1}}
def fails_once(s):
    calls['bad']+=1
    if calls['bad']==1: raise ValueError('injected failure')
    return {'results':{'bad':2}}
b=StateGraph(Parallel).add_node('good',succeeds).add_node('bad',fails_once)
b.add_edge(START,'good');b.add_edge(START,'bad');b.add_edge(['good','bad'],END)
pending=b.compile(checkpointer=InMemorySaver());config={'configurable':{'thread_id':'pending-writes'}}
try: pending.invoke({'results':{}},config)
except ValueError: pass
else: raise AssertionError('failure not observed')
recovered=pending.invoke(None,config)
assert recovered['results']=={'good':1,'bad':2} and calls=={'good':1,'bad':2}
print('重试次数:',attempts,'; pending writes 恢复次数:',calls)

## 6. Store 与线程、用户的边界
Store 的 namespace 是数据分区，不是认证系统。应用先从可信会话取 user_id，再构造 namespace；不能让模型选用户。InMemoryStore 也不会跨进程保存。

In [ ]:
store=InMemoryStore()
store.put(('alice','preferences'),'language',{'value':'中文','source':'explicit user choice'})
assert store.get(('alice','preferences'),'language').value['value']=='中文'
assert store.get(('bob','preferences'),'language') is None
assert InMemoryStore().get(('alice','preferences'),'language') is None
from dataclasses import dataclass
from langgraph.runtime import Runtime
@dataclass
class Context:
    user_id:str
class ContextState(TypedDict):
    request_user:str
    language:str
def read_preference(s,runtime:Runtime[Context]):
    item=runtime.store.get((runtime.context.user_id,'preferences'),'language')
    return {'language':item.value['value'] if item else '未设置'}
ctx=StateGraph(ContextState,context_schema=Context).add_node('read',read_preference).add_edge(START,'read').add_edge('read',END).compile(store=store)
# 输入假称 alice，可信上下文仍为 bob。
assert ctx.invoke({'request_user':'alice'},context=Context('bob'))['language']=='未设置'
assert ctx.invoke({'request_user':'bob'},context=Context('alice'))['language']=='中文'
print('namespace 与可信 runtime context 的边界已验证')

## 7. 留下自己的实验记录
先修改一个分支、一个 tool call、一个过滤字段和一次失败次数，记录预测与观察。报告记录本机 SDK 机制断言，不是模型、云服务或生产安全认证。将 JSON 导回阅读页，保留你原有笔记。

In [ ]:
from importlib.metadata import version
from pathlib import Path
import json
checks={'exclusive_route':good==['choose','search'],'static_edge_counterexample':set(bad)=={'choose','search','answer'},'send_join':batch['total']==29 and batch['joins']==1,'tool_result':messages[2].content=='42','private_stream_visible':any('internal_note' in e['data'] for e in events if e['type']=='values'),'filtered_stream':all(set(e['data'])<={'answer'} for e in safe),'retry_attempts':attempts['n']==3,'pending_writes':calls=={'good':1,'bad':2},'trusted_context':ctx.invoke({'request_user':'alice'},context=Context('bob'))['language']=='未设置'}
report={'experiment':'langgraph-control-flow','model_called':False,'version':version('langgraph'),'checks':checks,'my_explanation':''}
assert all(checks.values())
Path('langgraph_flow_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))